In [ ]:
!pip install ultralytics==8.2.0
!pip install opencv-python
!pip install pandas scikit-learn joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.8/750.8 kB 54.7 MB/s eta 0:00:00


In [ ]:
import torch

# PATCH FOR TORCH 2.6
original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return original_load(*args, **kwargs)

torch.load = patched_load

from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")

print("✅ YOLO WORKING")

100%|██████████| 6.51M/6.51M [00:00<00:00, 387MB/s]

✅ YOLO WORKING


In [ ]:
from google.colab import files

files.upload()

Saving cleaned_acc dataset.csv to cleaned_acc dataset.csv


{'cleaned_acc dataset.csv': b'left_hip_x,left_hip_y,right_hip_x,right_hip_y,left_knee_x,left_knee_y,right_knee_x,right_knee_y,left_ankle_x,left_ankle_y,right_ankle_x,right_ankle_y,label\n0.764494001865387,0.5547617077827454,0.7954400777816772,0.5531736016273499,0.7573732137680054,0.6482534408569336,0.8034689426422119,0.6605230569839478,0.7238406538963318,0.7416420578956604,0.7793617844581604,0.7584294080734253,2\n0.4371691346168518,0.4519676268100738,0.4893019497394562,0.4527346491813659,0.3847932517528534,0.4974299073219299,0.5333892107009888,0.4902978539466858,0.3792335391044616,0.5536743998527527,0.5462018251419067,0.5465031266212463,0\n0.4226000607013702,0.5439444780349731,0.4802448749542236,0.5572141408920288,0.3687704205513,0.6043437719345093,0.523460328578949,0.6098116040229797,0.3120018541812897,0.6677823662757874,0.5045884251594543,0.6727002263069153,1\n0.8554152846336365,0.5885378122329712,0.7716264724731445,0.5849727988243103,0.883670449256897,0.6656331419944763,0.7637991309

In [ ]:
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# LOAD DATASET
df = pd.read_csv("cleaned_acc dataset.csv")

print(df.head())

# FEATURES
X = df.drop("label", axis=1)

# LABELS
y = df["label"]

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# TRAIN RF
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

# TEST
preds = rf_model.predict(X_test)

acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

# SAVE PKL
joblib.dump(
    rf_model,
    "footwork_rf_model.pkl"
)

print("✅ PKL SAVED")

   left_hip_x  left_hip_y  right_hip_x  right_hip_y  left_knee_x  left_knee_y  \
0    0.764494    0.554762     0.795440     0.553174     0.757373     0.648253   
1    0.437169    0.451968     0.489302     0.452735     0.384793     0.497430   
2    0.422600    0.543944     0.480245     0.557214     0.368770     0.604344   
3    0.855415    0.588538     0.771626     0.584973     0.883670     0.665633   
4    0.592616    0.581089     0.638776     0.585053     0.565854     0.652912   

   right_knee_x  right_knee_y  left_ankle_x  left_ankle_y  right_ankle_x  \
0      0.803469      0.660523      0.723841      0.741642       0.779362   
1      0.533389      0.490298      0.379234      0.553674       0.546202   
2      0.523460      0.609812      0.312002      0.667782       0.504588   
3      0.763799      0.670546      0.910101      0.730436       0.786051   
4      0.675894      0.676501      0.509597      0.722586       0.691068   

   right_ankle_y  label  
0       0.758429      2  
1   

In [ ]:
import cv2
import numpy as np
import pandas as pd
import joblib

from ultralytics import YOLO
from google.colab import files

# =========================
# LOAD MODELS
# =========================

rf_model = joblib.load(
    "footwork_rf_model.pkl"
)

model = YOLO("yolov8n-pose.pt")

# =========================
# LABELS
# =========================

label_map = {
    0: "Forehand Front",
    1: "Forehand Mid",
    2: "Forehand Back",
    3: "Backhand Front",
    4: "Backhand Mid",
    5: "Backhand Back"
}

# =========================
# UPLOAD VIDEO
# =========================

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

# =========================
# VIDEO SETUP
# =========================

cap = cv2.VideoCapture(video_path)

width = int(cap.get(3))
height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(
    "final_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

# =========================
# TRACKING VARIABLES
# =========================

player_box = None

alpha = 0.7

frame_count = 0

# =========================
# PROCESS VIDEO
# =========================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # PROCESS EVERY 3RD FRAME
    if frame_count % 3 != 0:

        out.write(frame)
        continue

    results = model(
        frame,
        verbose=False
    )

    detections = []

    for r in results:

        if r.boxes is None:
            continue

        boxes = r.boxes.xyxy.cpu().numpy()

        keypoints = r.keypoints.xy.cpu().numpy()

        for i in range(len(boxes)):

            x1, y1, x2, y2 = map(
                int,
                boxes[i]
            )

            area = (x2-x1)*(y2-y1)

            detections.append(
                (
                    area,
                    (x1,y1,x2,y2),
                    keypoints[i]
                )
            )

    # =========================
    # BIGGEST PLAYER ONLY
    # =========================

    if detections:

        detections.sort(reverse=True)

        _, box, kpts = detections[0]

        # SMOOTH TRACKING
        if player_box is None:

            player_box = box

        else:

            player_box = tuple(
                int(alpha*p + (1-alpha)*n)
                for p,n in zip(player_box, box)
            )

        x1,y1,x2,y2 = player_box

        # SMART PADDING
        pad_x = int((x2-x1)*0.08)
        pad_y = int((y2-y1)*0.12)

        x1 = max(0, x1-pad_x)
        y1 = max(0, y1-pad_y)
        x2 = min(width, x2+pad_x)
        y2 = min(height, y2+pad_y)

        # =========================
        # LOWER BODY KEYPOINTS
        # =========================

        selected = [
            11,12,
            13,14,
            15,16
        ]

        features = []

        for idx in selected:

            features.append(
                kpts[idx][0] / width
            )

            features.append(
                kpts[idx][1] / height
            )

        features = np.array(
            features
        ).reshape(1,-1)

        features = pd.DataFrame(
            features,
            columns=rf_model.feature_names_in_
        )

        # =========================
        # RF PREDICTION
        # =========================

        pred = rf_model.predict(features)[0]

        prob = rf_model.predict_proba(features)

        conf = np.max(prob)

        class_name = label_map.get(
            pred,
            str(pred)
        )

        label = f"{class_name} ({conf*100:.1f}%)"

        # =========================
        # DRAW BOX
        # =========================

        cv2.rectangle(
            frame,
            (x1,y1),
            (x2,y2),
            (0,255,0),
            2
        )

        # DRAW LABEL
        cv2.putText(
            frame,
            label,
            (x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0,255,0),
            2
        )

    out.write(frame)

    # Progress
    if frame_count % 100 == 0:
        print("Processed:", frame_count)

# =========================
# FINISH
# =========================

cap.release()
out.release()

print("✅ FINAL VIDEO READY")

files.download("final_output.mp4")

Saving srikanth k trim.mp4 to srikanth k trim.mp4
Processed: 300
Processed: 600
Processed: 900
Processed: 1200
✅ FINAL VIDEO READY


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>